# Phase 6: SOTA Push — Stage A (Clean Re-run)

**Stage A**: Italian PD only | 1 seed (42) | 5 folds | 4 models × 5 folds = **20 runs**

| Model | Role |
|---|---|
| `resnet50` | Baseline (no attention) |
| `resnet50_ca` | Coordinate Attention |
| `resnet50_ca_lstm` | CA + BiLSTM temporal head |
| `dual_cnn_lstm` | Dual CNN + BiLSTM fusion |

### Goal
Clean re-run of Stage A to replace results contaminated by stale Kaggle dataset files.
Inline leakage guard will halt training if any subject overlap is detected.

In [ ]:
import os
import shutil
import zipfile
import subprocess
import sys
from pathlib import Path

# ── Hardcoded dataset path — avoids stale-file bugs from Kaggle versioning ──
DATASET_SLUG = 'phase6-italian-pd-clean'
KAGGLE_INPUT_DIR = Path('/kaggle/input')
EXPECTED_DIR = KAGGLE_INPUT_DIR / DATASET_SLUG
WORK_DIR = Path('/kaggle/working/PROJECT')

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Look for a zip first, then assume auto-extracted
zips = list(EXPECTED_DIR.glob('*.zip'))

if zips:
    ZIP_PATH = zips[0]
    print(f'Found zip: {ZIP_PATH}')
    print('Extracting... (may take 2-3 mins)')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('/kaggle/working/')
    print('Done extracting.')
else:
    print(f'No zip found — copying from {EXPECTED_DIR}...')
    if not EXPECTED_DIR.exists():
        raise FileNotFoundError(
            f'Dataset not found at {EXPECTED_DIR}! '
            f'Make sure the Kaggle dataset slug is "{DATASET_SLUG}"'
        )
    shutil.copytree(EXPECTED_DIR, WORK_DIR, dirs_exist_ok=True)
    print('Done copying.')

In [ ]:
os.chdir(WORK_DIR)
print(f'CWD: {os.getcwd()}')

ROOT = WORK_DIR
SPLITS = ROOT / 'product/artifacts/splits'

# ── Leakage guard: assert 0 subject overlap for every fold ──
import pandas as pd
leakage_found = False
for fold in range(5):
    train_csv = SPLITS / f'train_italian_fold{fold}.csv'
    val_csv = SPLITS / f'val_italian_fold{fold}.csv'
    if not train_csv.exists() or not val_csv.exists():
        print(f'WARN: fold{fold} CSVs missing, skipping check')
        continue
    t_subj = set(pd.read_csv(train_csv)['subject_id'])
    v_subj = set(pd.read_csv(val_csv)['subject_id'])
    overlap = t_subj & v_subj
    if overlap:
        print(f'CRITICAL: fold{fold} has {len(overlap)} overlapping subjects: {overlap}')
        leakage_found = True
    else:
        print(f'fold{fold}: OK (train={len(t_subj)} subjects, val={len(v_subj)} subjects, overlap=0)')

if leakage_found:
    raise RuntimeError('DATA LEAKAGE DETECTED — aborting. Delete and re-upload the Kaggle dataset.')
print('All folds passed leakage check.')

In [ ]:
import shutil
runs_root = ROOT / "product/artifacts/runs"
cleaned = 0
PHASE6_TAGS = ["_resnet50_", "_resnet50_ca_", "_resnet50_ca_lstm_", "_dual_cnn_lstm_"]
for ds in ["italian_pd"]:
    ds_dir = runs_root / ds
    if not ds_dir.exists():
        continue
    for run_dir in list(ds_dir.iterdir()):
        name = run_dir.name
        if any(tag in f"_{name}_" for tag in PHASE6_TAGS):
            shutil.rmtree(run_dir)
            cleaned += 1
print(f"Cleaned {cleaned} stale Phase 6 run directories.")

In [ ]:
# ============================================================
# Phase 6 Stage B: Italian PD, 3 seeds, 5 folds
# 2 models × 3 seeds × 5 folds = 30 runs
# Goal: Statistical validation of dual_cnn_sa_lstm and resnet50_ca_lstm
# ============================================================
import sys
import subprocess
from pathlib import Path

ROOT = Path('/kaggle/working/PROJECT')
total = 20
done = 0
print(f"Phase 6 Stage B: {total} runs (4 models × 1 seeds × 5 folds on italian_pd)")


run_name = "italian_pd_resnet50_s42_fold0"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "resnet50",
        "--fold", "0",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_resnet50_s42_fold0",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_resnet50_s42_fold1"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "resnet50",
        "--fold", "1",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_resnet50_s42_fold1",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_resnet50_s42_fold2"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "resnet50",
        "--fold", "2",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_resnet50_s42_fold2",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_resnet50_s42_fold3"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "resnet50",
        "--fold", "3",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_resnet50_s42_fold3",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_resnet50_s42_fold4"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "resnet50",
        "--fold", "4",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_resnet50_s42_fold4",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_resnet50_ca_s42_fold0"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "resnet50_ca",
        "--fold", "0",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_resnet50_ca_s42_fold0",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_resnet50_ca_s42_fold1"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "resnet50_ca",
        "--fold", "1",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_resnet50_ca_s42_fold1",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_resnet50_ca_s42_fold2"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "resnet50_ca",
        "--fold", "2",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_resnet50_ca_s42_fold2",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_resnet50_ca_s42_fold3"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "resnet50_ca",
        "--fold", "3",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_resnet50_ca_s42_fold3",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_resnet50_ca_s42_fold4"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "resnet50_ca",
        "--fold", "4",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_resnet50_ca_s42_fold4",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_resnet50_ca_lstm_s42_fold0"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "resnet50_ca_lstm",
        "--fold", "0",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_resnet50_ca_lstm_s42_fold0",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_resnet50_ca_lstm_s42_fold1"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "resnet50_ca_lstm",
        "--fold", "1",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_resnet50_ca_lstm_s42_fold1",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_resnet50_ca_lstm_s42_fold2"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "resnet50_ca_lstm",
        "--fold", "2",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_resnet50_ca_lstm_s42_fold2",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_resnet50_ca_lstm_s42_fold3"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "resnet50_ca_lstm",
        "--fold", "3",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_resnet50_ca_lstm_s42_fold3",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_resnet50_ca_lstm_s42_fold4"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "resnet50_ca_lstm",
        "--fold", "4",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_resnet50_ca_lstm_s42_fold4",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_dual_cnn_lstm_s42_fold0"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "dual_cnn_lstm",
        "--fold", "0",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_dual_cnn_lstm_s42_fold0",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_dual_cnn_lstm_s42_fold1"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "dual_cnn_lstm",
        "--fold", "1",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_dual_cnn_lstm_s42_fold1",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_dual_cnn_lstm_s42_fold2"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "dual_cnn_lstm",
        "--fold", "2",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_dual_cnn_lstm_s42_fold2",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_dual_cnn_lstm_s42_fold3"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "dual_cnn_lstm",
        "--fold", "3",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_dual_cnn_lstm_s42_fold3",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

run_name = "italian_pd_dual_cnn_lstm_s42_fold4"
summary_path = ROOT / f"product/artifacts/runs/italian_pd/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "italian_pd",
        "--model_type", "dual_cnn_lstm",
        "--fold", "4",
        "--epochs", "20",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "italian_pd_dual_cnn_lstm_s42_fold4",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.0",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")

In [ ]:
# Aggregate Stage A results into a summary table
import glob

print("\n=== Phase 6 Stage A Results (Italian PD) ===\n")
print(f"{'Model':<25} {'Fold':<6} {'Best F1':>8} {'Final F1':>9} {'AUC':>7}")
print("-" * 60)

import json as _json
from pathlib import Path as _P

runs_root = ROOT / "product/artifacts/runs/italian_pd"
for model_type, _, _ in EXPERIMENTS:
    fold_f1s = []
    for fold in range(5):
        run_name = f"italian_pd_{model_type}_s42_fold{fold}"
        summary_p = runs_root / run_name / "summary.json"
        if summary_p.exists():
            s = _json.loads(summary_p.read_text())
            f1  = s.get("best_macro_f1", 0.0)
            ff1 = s.get("final_macro_f1", 0.0)
            auc = s.get("final_auc", 0.0)
            fold_f1s.append(f1)
            print(f"{model_type:<25} {fold:<6} {f1:>8.4f} {ff1:>9.4f} {auc:>7.4f}")
        else:
            print(f"{model_type:<25} {fold:<6} {'MISSING':>8}")
    if fold_f1s:
        import numpy as np
        print(f"  → mean ± std: {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")
    print()

In [ ]:
# Zip all results for download
output_zip = '/kaggle/working/phase6_results.zip'
print(f'Zipping results...')

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    runs_dir = ROOT / 'product/artifacts/runs'
    for f in runs_dir.glob('**/*'):
        if f.is_file() and f.suffix != '.pt':  # skip large model weights
            zipf.write(f, f.relative_to(runs_dir.parent.parent))

print(f'Done! Download phase6_results.zip from the Output tab.')
print(f'Stage A complete: 15 runs on Italian PD.')
print(f'EXIT CRITERIA: resnet50_ca_lstm must achieve F1 > 0.930 to proceed to Stage B.')